In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

os.environ["HADOOP_USER_NAME"] = "root"

spark = SparkSession.builder \
    .appName('Titanic') \
    .master('yarn') \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-namenode:9000") \
    .config("spark.hadoop.yarn.resourcemanager.hostname", "resourcemanager") \
    .config("spark.hadoop.yarn.resourcemanager.address", "resourcemanager:8032") \
    .config("spark.hadoop.yarn.resourcemanager.scheduler.address", "resourcemanager:8030") \
    .config("spark.driver.host", "172.30.1.13") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.executor.memory", "512m") \
    .config("spark.yarn.am.memory", "512m")\
    .getOrCreate()

In [3]:
bronze_path = "hdfs://hadoop-namenode:9000/user/root/datalake/bronze/sensor_data/"
gold_path = "hdfs://hadoop-namenode:9000/user/root/datalake/gold/"

df = spark.read.parquet(bronze_path)

df.show(5)
print(df.count())

+----+-----+--------+-----+--------------------+-----+-----------+------+------+-----+--------+-----------------+-------------------+----------------+
| Age|Cabin|Embarked| Fare|                Name|Parch|PassengerId|Pclass|   Sex|SibSp|Survived|           Ticket|         event_time|          source|
+----+-----+--------+-----+--------------------+-----+-----------+------+------+-----+--------+-----------------+-------------------+----------------+
|28.0|  C52|       S|26.55|Bjornstrom-Steffa...|    0|        431|     1|  male|    0|       1|           110564|2026-04-28 22:33:49|SIMULATED_DEVICE|
|NULL| NULL|       S| 16.1|Thorneycroft, Mrs...|    0|        432|     3|female|    1|       1|           376564|2026-04-28 22:33:49|SIMULATED_DEVICE|
|42.0| NULL|       S| 26.0|Louch, Mrs. Charl...|    0|        433|     2|female|    1|       1|       SC/AH 3085|2026-04-28 22:33:49|SIMULATED_DEVICE|
|17.0| NULL|       S|7.125|Kallio, Mr. Nikol...|    0|        434|     3|  male|    0|       0

In [4]:
df = df.dropDuplicates()

In [5]:
from pyspark.sql.types import DoubleType

df = df.withColumn("Age", F.col("Age").cast(DoubleType()))
age_median = df.approxQuantile("Age", [0.5], 0.01)[0]
df = df.fillna({"Age": age_median})

In [6]:
df = df.fillna({"Embarked": "S"})

In [7]:
fare_mean = df.select(F.mean("Fare")).collect()[0][0]
df = df.fillna({"Fare": fare_mean})

In [8]:
df = df.drop("Cabin")

In [9]:
from pyspark.sql.types import IntegerType, DoubleType
df = df.withColumn("Age", df["Age"].cast(DoubleType())) \
       .withColumn("Fare", df["Fare"].cast(DoubleType())) \
       .withColumn("Survived", df["Survived"].cast(IntegerType())) \
       .withColumn("Pclass", df["Pclass"].cast(IntegerType()))

In [10]:
df = df.withColumn("Sex", F.lower(F.col("Sex"))) \
       .withColumn("Embarked", F.upper(F.col("Embarked")))

In [11]:
df = df.withColumn(
    "AgeGroup",
    F.when(df.Age < 18, "Child")
     .when(df.Age < 60, "Adult")
     .otherwise("Senior")
)

In [12]:
df = df.drop("Ticket")

In [13]:
df.printSchema()
df.show(5)
print("Final Count:", df.count())

root
 |-- Age: double (nullable = false)
 |-- Embarked: string (nullable = false)
 |-- Fare: double (nullable = false)
 |-- Name: string (nullable = true)
 |-- Parch: long (nullable = true)
 |-- PassengerId: long (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Sex: string (nullable = true)
 |-- SibSp: long (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- event_time: string (nullable = true)
 |-- source: string (nullable = true)
 |-- AgeGroup: string (nullable = false)

+----+--------+--------+--------------------+-----+-----------+------+------+-----+--------+-------------------+----------------+--------+
| Age|Embarked|    Fare|                Name|Parch|PassengerId|Pclass|   Sex|SibSp|Survived|         event_time|          source|AgeGroup|
+----+--------+--------+--------------------+-----+-----------+------+------+-----+--------+-------------------+----------------+--------+
|30.0|       S|    93.5|Perreault, Miss. ...|    0|        521|     1|female|   

In [14]:
df = df.withColumn(
    "FamilySize",
    F.col("SibSp").cast("int") + F.col("Parch").cast("int") + F.lit(1)
)

In [15]:
df = df.withColumn(
    "IsAlone",
    F.when(F.col("FamilySize") == 1, 1).otherwise(0)
)

In [16]:
df = df.withColumn(
    "FarePerPerson",
    F.col("Fare") / F.col("FamilySize")
)

In [17]:
df = df.withColumn(
    "AgeBucket",
    F.when(F.col("Age") < 12, "Child")
     .when(F.col("Age") < 18, "Teen")
     .when(F.col("Age") < 60, "Adult")
     .otherwise("Senior")
)

In [18]:
df = df.withColumn("Sex", F.lower(F.col("Sex"))) \
       .withColumn("Embarked", F.upper(F.col("Embarked")))

In [19]:
df = df.withColumnRenamed("SibSp", "SiblingsSpouses") \
       .withColumnRenamed("Parch", "ParentsChildren")

In [20]:
sex_survival = df.groupBy("Sex").agg(
    F.count("*").alias("Total"),
    F.sum("Survived").alias("Survived"),
    (F.sum("Survived") / F.count("*")).alias("SurvivalRate")
)

In [21]:
class_survival = df.groupBy("Pclass").agg(
    F.count("*").alias("Total"),
    F.sum("Survived").alias("Survived")
)

In [22]:
age_survival = df.groupBy("AgeBucket").agg(
    F.count("*").alias("Total"),
    F.sum("Survived").alias("Survived")
)

In [23]:
fact_passenger = df.select(
    "PassengerId",
    "Survived",
    "Pclass",
    "Fare",
    "FarePerPerson",
    "FamilySize",
    "IsAlone",
    "Age"
)

In [24]:
dim_passenger = df.select(
    "PassengerId",
    "Name",
    "Sex",
    "AgeBucket"
)

In [25]:
dim_embarked = df.select(
    "PassengerId",
    "Embarked"
)

In [26]:
dim_class = df.select(
    "PassengerId",
    "Pclass"
)

In [27]:
df.show(5)
df.printSchema()

+----+--------+--------+--------------------+---------------+-----------+------+------+---------------+--------+-------------------+----------------+--------+----------+-------+-------------+---------+
| Age|Embarked|    Fare|                Name|ParentsChildren|PassengerId|Pclass|   Sex|SiblingsSpouses|Survived|         event_time|          source|AgeGroup|FamilySize|IsAlone|FarePerPerson|AgeBucket|
+----+--------+--------+--------------------+---------------+-----------+------+------+---------------+--------+-------------------+----------------+--------+----------+-------+-------------+---------+
|30.0|       S|    93.5|Perreault, Miss. ...|              0|        521|     1|female|              0|       1|2026-04-28 22:34:16|SIMULATED_DEVICE|   Adult|         1|      1|         93.5|    Adult|
|22.0|       C|    49.5|Frolicher, Miss. ...|              2|        540|     1|female|              0|       1|2026-04-28 22:34:17|SIMULATED_DEVICE|   Adult|         3|      0|         16.5| 

In [28]:
fact_passenger.write.mode("overwrite").parquet(f"{gold_path}fact_passenger")

In [29]:
dim_passenger.write.mode("overwrite").parquet(f"{gold_path}dim_passenger")

In [30]:
dim_embarked.write.mode("overwrite").parquet(f"{gold_path}dim_embarked")

In [31]:
dim_class.write.mode("overwrite").parquet(f"{gold_path}dim_class")